In [1]:
import numpy as np
import qiskit.quantum_info as qi
from processtensor import QTensor, QuantumState, UnitarySE, ProcessTensor
from qiskit.visualization import plot_state_hinton

# Bond dim vs. random choi state size

In [2]:
I = np.eye(2)
X = qi.Pauli('X').to_matrix()
Y = qi.Pauli('Y').to_matrix()
Z = qi.Pauli('Z').to_matrix()

def random_pt(dS, dE):
    U1 = qi.random_unitary(dS * dE).data
    U2 = qi.random_unitary(dS * dE).data
    q1 = UnitarySE.from_unitary(U1, dS, dE)
    q2 = UnitarySE.from_unitary(U2, dS, dE)
    pt = q1 @ q2
    pt = pt.trace_subsystem(0, pt.ndim - 1)
    pt = ProcessTensor(pt)
    return pt

def pauli_superop_tensor(d):
    n_qubits = int(np.log2(d))
    paulis = [p.to_matrix() for p in qi.pauli_basis(n_qubits)]
    return np.array([np.kron(p.T, p) for p in paulis])

def multitime_twirl(choi, d=2):
    d2 = d * d
    pst = pauli_superop_tensor(d)
    pt_data = ProcessTensor.from_choi(choi, (d, d, d, d)).data.reshape(d2, d2, d2, d2)
    twirled = np.einsum('abcd, xai, xbj, yck, ydl -> ijkl', pt_data, pst, pst, pst, pst)

    return ProcessTensor(twirled.reshape(1, d2, d2, d2, d2, 1)).choi_matrix

def multitime_twirl_alt(pt_choi):
    """"""
    n_qubits = int(np.log2(pt_choi.shape[0]) / 2)  # Number of system qubits
    twirled = np.zeros_like(pt_choi, dtype=np.complex128)
    factor = 1 / (4 ** n_qubits)  # Normalization factor for
    
    basis_strings = qi.pauli_basis(n_qubits)
    basis_superop_products = [superop_from_pauli_string(basis_string) for basis_string in basis_strings]
    
    for superop_product in basis_superop_products:
        
        twirled += factor * superop_product @ pt_choi @ superop_product.conj().T
        
    return twirled

def superop_from_pauli_string(pauli_string):
    n_qubits = len(pauli_string)
    
    superops = []
    
    for i in range(n_qubits):
        matrix = pauli_string[i].to_matrix()
        superop = np.kron(matrix, np.conj(matrix.T))
        superops.append(superop)
        
    return multi_kron(superops)

def multi_kron(Ms):
    """Compute the tensor product of a list of matrices."""
    prod = Ms[0]
    for i in range(1, len(Ms)):
        prod = np.kron(prod, Ms[i])
    return prod

In [3]:
def process_tensor_from_stinespring(unitary, dS, dE):
    q1 = UnitarySE.from_unitary(unitary, dS, dE)
    q2 = q1 @ q1
    pt = q2.trace_subsystem(0, q2.ndim - 1)
    pt = ProcessTensor(pt)
    return pt

def twirl_superop(superop):
    """Twirls a system-environment superoperator over the Pauli group on the system."""
    dS = int(np.sqrt(superop.shape[1]))
    pst = pauli_superop_tensor(dS)
    return np.einsum('abcd, xbj, xck -> ajkd', superop, pst, pst)

def unitary_svd(unitary, dS, dE):
    shuffled = unitary.reshape(dE, dS, dE, dS).transpose(0, 2, 1, 3).reshape(dE*dE, dS*dS)
    U, S, Vh = np.linalg.svd(shuffled)
    n_nonzero = np.sum(S > 1e-10)
    U = U[:, :n_nonzero]
    S = S[:n_nonzero]
    Vh = Vh[:n_nonzero, :]
    V = np.diag(S) @ Vh
    return U, S, V

def superop_svd(superop, dS, dE):
    dE2 = dE * dE
    dS2 = dS * dS
    shuffled = superop.transpose(0, 3, 1, 2).reshape(dE**4, dS**4)
    U, S, Vh = np.linalg.svd(shuffled)
    n_nonzero = np.sum(S > 1e-10)
    U = U[:, :n_nonzero]
    S = S[:n_nonzero]
    Vh = Vh[:n_nonzero, :]
    V = np.diag(S) @ Vh
    return U, S, V

def choi_svd(choi):
    d = int(np.sqrt(choi.shape[0]))
    shuffled = choi.reshape(d, d, d, d).transpose(0, 2, 1, 3).reshape(d*d, d*d)
    U, S, Vh = np.linalg.svd(shuffled)
    n_nonzero = np.sum(S > 1e-10)
    U = U[:, :n_nonzero]
    S = S[:n_nonzero]
    Vh = Vh[:n_nonzero, :]
    V = np.diag(S) @ Vh
    return U, S, V

In [4]:
def analyse_choi(choi, name=""):
    choi_twirled = multitime_twirl(choi)

    _, S1, _ = choi_svd(choi)
    _, S2, _ = choi_svd(choi_twirled)

    print(f"PT MPO bond dimension: {len(S1)}")
    print(f"SPC MPS bond dimension: {len(S2)}")

def analyse_unitary(unitary, dS, dE, name=""):
    
    superop = UnitarySE.from_unitary(unitary, dS, dE).data
    superop_twirled = twirl_superop(superop)

    _, S1, _ = unitary_svd(unitary, dS, dE)
    _, S2, _ = superop_svd(superop, dS, dE)
    _, S3, _ = superop_svd(superop_twirled, dS, dE)

    print(f"Unitary Schmidt rank: {len(S1)}")
    print(f"Superoperator Schmidt rank: {len(S2)}")
    print(f"Superoperator Schmidt rank (Twirled): {len(S3)}")

### Random ($16\times 16$) Choi matrix

In [5]:
dS = 2
choi1 = qi.random_density_matrix(dS**4).data
choi1_twirled = multitime_twirl(choi1)

analyse_choi(choi1, "Random Choi state")

PT MPO bond dimension: 16
SPC MPS bond dimension: 4


### Process tensor from random unitaries ($k = 2, d_S = 2, d_E = 2$)

In [6]:
dS = 2
dE = 2
unitary2 = qi.random_unitary(dS * dE).data
pt2 = process_tensor_from_stinespring(unitary2, dS, dE)
choi2 = pt2.choi_matrix

analyse_unitary(unitary2, dS, dE, name=f"Random unitary (dS={dS}, dE={dE})")
analyse_choi(choi2, f"PT Choi from random unitaries (k=2, dS={dS}, dE={dE})")

Unitary Schmidt rank: 4
Superoperator Schmidt rank: 16
Superoperator Schmidt rank (Twirled): 4
PT MPO bond dimension: 4
SPC MPS bond dimension: 4


### Process tensor from random unitaries ($k = 2, d_S=2, d_E=3$)

In [7]:
dS = 2
dE = 3
unitary3 = qi.random_unitary(dS * dE).data
pt3 = process_tensor_from_stinespring(unitary3, dS, dE)
choi3 = pt3.choi_matrix

analyse_unitary(unitary3, dS, dE, name=f"Random unitary (dS={dS}, dE={dE})")
analyse_choi(choi3, f"PT Choi from random unitaries (k=2, dS={dS}, dE={dE})")

Unitary Schmidt rank: 4
Superoperator Schmidt rank: 16
Superoperator Schmidt rank (Twirled): 4
PT MPO bond dimension: 9
SPC MPS bond dimension: 4


### Process tensor from random unitaries ($k=2, d_S=2, d_E=4$)

In [8]:
dS = 2
dE = 4
unitary4 = qi.random_unitary(dS * dE).data
pt4 = process_tensor_from_stinespring(unitary4, dS, dE)
choi4 = pt4.choi_matrix

analyse_unitary(unitary4, dS, dE, name=f"Random unitary (dS={dS}, dE={dE})")
analyse_choi(choi4, f"PT Choi from random unitaries (k=2, dS={dS}, dE={dE})")

Unitary Schmidt rank: 4
Superoperator Schmidt rank: 16
Superoperator Schmidt rank (Twirled): 4
PT MPO bond dimension: 13
SPC MPS bond dimension: 4


### Environment controlled CNOT

In [9]:
dS = 2
dE = 2
CNOT = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]])
pt5 = process_tensor_from_stinespring(CNOT, dS, dE)
choi5 = pt5.choi_matrix

analyse_unitary(CNOT, dS, dE, name=f"CNOT (dS={dS}, dE={dE})")
analyse_choi(choi5, f"PT Choi from CNOT (k=2, dS={dS}, dE={dE})")

Unitary Schmidt rank: 2
Superoperator Schmidt rank: 4
Superoperator Schmidt rank (Twirled): 2
PT MPO bond dimension: 2
SPC MPS bond dimension: 2


### SWAP

In [10]:
dS = 2
dE = 2
SWAP = np.array([[1, 0, 0, 0],
                 [0, 0, 1, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1]])
pt6 = process_tensor_from_stinespring(SWAP, dS, dE)
choi6 = pt6.choi_matrix
analyse_unitary(SWAP, dS, dE, name=f"SWAP (dS={dS}, dE={dE})")
analyse_choi(choi6, f"PT Choi from SWAP (k=2, dS={dS}, dE={dE})")

Unitary Schmidt rank: 4
Superoperator Schmidt rank: 16
Superoperator Schmidt rank (Twirled): 4
PT MPO bond dimension: 4
SPC MPS bond dimension: 1


### Pauli Product

In [11]:
dS = 2
dE = 2
pauli_prod = np.kron(X, Y)
pt7 = process_tensor_from_stinespring(pauli_prod, dS, dE)
choi7 = pt7.choi_matrix
analyse_unitary(pauli_prod, dS, dE, name=f"Pauli Product (dS={dS}, dE={dE})")
analyse_choi(choi7, f"PT Choi from Pauli Product (k=2, dS={dS}, dE={dE})")

Unitary Schmidt rank: 1
Superoperator Schmidt rank: 1
Superoperator Schmidt rank (Twirled): 1
PT MPO bond dimension: 1
SPC MPS bond dimension: 1
